# Phase 1 — The Core Model

**Goal:** Define the cost-effectiveness ratio, introduce the internal metric $v_i$ (SD per dollar), and build the unadjusted league table.  
No uncertainty, no time dimension. Everything is point estimates.

## Key concepts

**Internal metric (effectiveness):** $v_i = \frac{\hat{\tau}_i}{C_i}$  
where $\hat{\tau}_i$ is the estimated effect in SD and $C_i$ is cost per beneficiary in USD.

**Display metric (CE ratio):** $CE_i^* = \frac{0.1}{v_i} = \frac{0.1 \cdot C_i}{\hat{\tau}_i}$  
This is read as \"cost per 0.1 SD of learning gain.\"

**Assumptions active in Phase 1:**
- **A0:** Effect estimate is known (no sampling uncertainty)
- **A1:** Costs are proportional to beneficiaries (no fixed costs)
- **A2:** One-year time horizon
- **A3:** No effect decay

Note: The full `Intervention` dataclass includes fields for fixed costs, durations, and decay rates (with defaults that make them inactive), so the data schema is forward-compatible with Phases 2–5.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from cea_dev.core import Intervention, InterventionSet
from cea_dev.visualization import plot_league_table

pd.set_option("display.precision", 4)

## Load the synthetic intervention data

The eight education interventions are parameterized from published evaluation ranges. All cost bounds are imputed at $\pm 20\%$ of the point estimate (cost_imputed = True for all).

In [ ]:
DATA_DIR = Path("../data")
iset = InterventionSet.from_csv(str(DATA_DIR / "interventions.csv"))
print(f"Loaded {len(iset.interventions)} interventions")

## Unadjusted league table

Ranked by effectiveness $v_i$ (SD per dollar) descending. The `ce_display` column is cost per 0.1 SD — lower is better.

In [ ]:
table = iset.league_table()
table

## League table chart

Horizontal bar chart sorted by effectiveness. Cost per 0.1 SD on x-axis (log scale). Color-coded by quartile:
- **Best Buy:** top quartile (lowest CE*)
- **Good Buy:** second quartile
- **Promising:** third quartile
- **Weak:** bottom quartile (highest CE*)

In [ ]:
fig, ax = plot_league_table(table)
plt.show()

## Notes

1. **TaRL ranks first** with effectiveness $v = 0.18 / 4.00 = 0.045$ SD per dollar ($CE^* = \$2.22$ per 0.1 SD).
2. **CCT ranks last** with $v = 0.07 / 22.00 \approx 0.0032$ SD per dollar ($CE^* = \$31.43$ per 0.1 SD).
3. **ECD appears mid-table** ($v = 0.0088$, rank 5) despite the largest effect (0.22 SD) because its cost is high (\$25/beneficiary). Its ranking will improve materially in Phase 2 when its 10-year duration is accounted for.
4. **Deworming ranks second** ($v = 0.025$) due to its extremely low cost (\$2/beneficiary) despite a small effect (0.05 SD). However, its SE/effect ratio (0.80) means its ranking is fragile — Phase 3 will show $P(\text{harm}) > 5\%$.
5. **The ranking is provisional.** All four assumptions (A0–A3) will be relaxed in subsequent phases.